# Getting started with SpikingGamma

This notebook will help you to get started with SpikingGamma. Besides the spikinggamma package, only PyTorch is a strict requirement, but for this notebook you also need tonic.
It is highly recommended to use a GPU. Make sure you have installed the correct version of PyTorch for this.

In [1]:
import torch
import torch.nn as nn
import tonic
from torch.utils.data import DataLoader
from tqdm import tqdm
from spikinggamma.neuron import SpikingGammaNeuron
from spikinggamma.spike.sigma_delta import SigmaDeltaSpikeFunction
from spikinggamma.dendrite.split import SplitDendriticFunction
from spikinggamma.dendrite.synapse.linear import LinearPerNeuron
from spikinggamma.signal.default import DefaultSignalFunction
from spikinggamma.spike.base import SpikeConfig
from spikinggamma.spike.kernel.gamma import GammaKernelFunction, compute_linspace_bucket_decay_rates
from spikinggamma.spike.identity import IdentitySpikeFunction
from spikinggamma.utils.datasets.shd import get_shd_dataset
from pathlib import Path


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Loading the data
We use SHD as example, with 250 timesteps. The number of features is scaled down to 140. For performance reasons, caching is used through tonic's `DiskCachedDataset`. This makes the first epoch much slower.

In [2]:
project_dir = Path.cwd().parent
data_path = project_dir / "data"
cache_path = project_dir / "cache"
train_dataset, test_dataset, input_shape, n_outputs, T = get_shd_dataset(data_path=data_path, cache_path=cache_path, timestep_size=3600)

## Neuron structure

A neuron has three components: dendritic (synaptic) function, signal function, and spike function.  
The `SpikingGammaNeuron` class processes input in three steps:


1. First, the kernel components (**not** the spikes, they are not used for training) are passed to the dendritic function, which applies a synaptic transform (e.g., linear or convolution) to each kernel component independently. The results are then weighted per component and summed such that there is one dendritic output value per neuron.
2. Second, the dendritic output is processed by the signal function to produce a neuron signal. No spikes are emitted at this stage.
3. Finally, the neuron signal goes to the spike function, producing spikes and kernel values. This decomposes the neuron signal into kernel components for downstream processing.

Since the input data is not yet decomposed into kernel components, another input spike function must be provided for just this purpose.
The spike function may be skipped for the output layer to allow passing the continuous neuron signal directly to the loss function, as we show in this example.


### Tips for building a network
`SpikingGammaNeuron` can be used like any other layer. Just remember that it contains both the synaptic transformation and the spiking activation. It is possible to do more transformations on the neuron signal in between the neuron layers, more about this in a later example notebook. Remember to allow for resetting the network (and to call it), this is very important to do in between samples!

In [3]:
# Hyperparameters
n_kernels = 10
decay_factor = 0.1
spike_config = SpikeConfig(
    base_spike_threshold=0.3,
    use_dynamic_threshold=True
)
bucket_decay_rates = compute_linspace_bucket_decay_rates(n_kernels, decay_factor)

# A simple fully connected network with 2 hidden layers
class SpikingGammaNet(nn.Module):
    def __init__(self, 
                 input_shape, 
                 n_outputs,
                 n_kernels,
                 bucket_decay_rates,
                 spike_config):
        super().__init__()
        layer1_shape_out = (256,)
        layer2_shape_out = (256,)
        layer3_shape_out = (n_outputs,)

        self.input_fn = IdentitySpikeFunction(GammaKernelFunction(bucket_decay_rates), spike_config)
        self.hid1 = SpikingGammaNeuron(
            dendritic_fn=SplitDendriticFunction(
                layer1_shape_out,
                n_kernels=n_kernels,
                synaptic_function=LinearPerNeuron(input_shape[0], layer1_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(pre_act_fn=nn.LayerNorm(layer1_shape_out)),
            spike_fn=SigmaDeltaSpikeFunction(GammaKernelFunction(bucket_decay_rates), spike_config),
        )
        self.hid2 = SpikingGammaNeuron(
            dendritic_fn=SplitDendriticFunction(
                layer2_shape_out,
                n_kernels=n_kernels,
                synaptic_function=LinearPerNeuron(layer1_shape_out[0], layer2_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(pre_act_fn=nn.LayerNorm(layer2_shape_out)),
            spike_fn=SigmaDeltaSpikeFunction(GammaKernelFunction(bucket_decay_rates), spike_config),
        )
        self.out = SpikingGammaNeuron(
            dendritic_fn=SplitDendriticFunction(
                layer3_shape_out,
                n_kernels=n_kernels,
                synaptic_function=LinearPerNeuron(layer2_shape_out[0], layer3_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(pre_act_fn=nn.LayerNorm(layer3_shape_out)),
        )


    def reset(self):
        self.input_fn.reset()
        self.hid1.reset()
        self.hid2.reset()
        self.out.reset()

    def detach(self):
        self.input_fn.detach()
        self.hid1.detach()
        self.hid2.detach()
        self.out.detach()

    def forward(self, s_in):
        s_in = s_in.flatten(start_dim=1)
        y_hat_k_in, _ = self.input_fn(s_in)
        y_hat_k_hid1, s_hid1 = self.hid1(y_hat_k_in)
        y_hat_k_hid2, s_hid2 = self.hid2(y_hat_k_hid1)
        y_out = self.out(y_hat_k_hid2)
        return y_out

## Training the network
Training is fairly straightforward. The usual PyTorch / deep learning mechanisms apply. Remember we do online learning, so the loss is computed per timestep.

In [4]:
train_batch_size = 32
test_batch_size = 128
train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, collate_fn=tonic.collation.PadTensors(batch_first=False))
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, collate_fn=tonic.collation.PadTensors(batch_first=False))
model = SpikingGammaNet(input_shape, n_outputs, n_kernels, bucket_decay_rates, spike_config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [5]:
model.train()
n_epochs = 1
for _ in range(n_epochs):
    batch_iter = iter(train_dataloader)
    pbar = tqdm(batch_iter, total=len(train_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        optimizer.zero_grad()
        total_loss = 0
        for t in range(data.shape[0]):
            y = model(data[t])
            loss = loss_fn(y, labels) * 1/data.shape[0]
            loss.backward()
            total_loss += loss.item()

        optimizer.step()
        pbar.set_description(f"Loss: {total_loss:.4f}")

Loss: 1.5481: 100%|██████████| 255/255 [01:01<00:00,  4.16it/s]


Remember that the first epoch may be slower because of tonic's `DiskCachedDataset` that is enabled by default.

## Testing
For computing the accuracy, we can sum the output signal and take the index with the maximum.

In [6]:
batch_iter = iter(test_dataloader)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    pbar = tqdm(batch_iter, total=len(test_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        ys = [] 
        for t in range(data.shape[0]):
            y = model(data[t])
            ys.append(y)
        ys = torch.stack(ys, dim=0)
        predictions = ys.sum(dim=0).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

100%|██████████| 18/18 [00:02<00:00,  7.20it/s]

Test Accuracy: 69.92%


You can save and load the model through the usual PyTorch module functionality.

In [7]:
# Save
torch.save(model.state_dict(), "spiking_gamma_model.pth")

In [8]:
# Load
model = SpikingGammaNet(input_shape, n_outputs, n_kernels, bucket_decay_rates, spike_config).to(device)
model.load_state_dict(torch.load("spiking_gamma_model.pth"))

<All keys matched successfully>